## convert to json

In [1]:
import pandas as pd
import numpy as np
import json

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv("SmartHomeIoTNLU.csv")

# =========================================================
# STEP 1: BUILD EVENT STRUCTURE (DEVICE INTEGRITY)
# =========================================================

events = []

for eid, g in df.groupby("Event_ID"):

    first = g.iloc[0]

    events.append({

        "Event_ID": first["Event_ID"],
        "Intent_Type": first["Intent_Type"],
        "Command": first["Command"],
        "Scenario": first["Scenario"],
        "Location": first["Location"],

        "Context": {
            "Location": first["Location"],
            "Time": first["Time"],
            "Light": first["Light"],
            "Temperature": first["Temperature"],
            "Noise": first["Noise"],
            "Occupancy": first["Occupancy"]
        },

        "DeviceActions": g[
            ["Device", "Action", "Parameters"]
        ].to_dict("records")
    })

df_events = pd.DataFrame(events)

# =========================================================
# STEP 2: DEFINE GLOBAL SEMANTIC KEY
# =========================================================
# THIS IS CRITICAL: no stratified splitting before this

df_events["SplitKey"] = df_events.apply(
    lambda r: f"{r['Command']}||{r['Location']}",
    axis=1
)

# =========================================================
# STEP 3: GLOBAL SPLIT (NO STRATIFICATION HERE)
# =========================================================

np.random.seed(42)

unique_keys = df_events["SplitKey"].unique()
unique_keys = np.array(unique_keys)

np.random.shuffle(unique_keys)

n = len(unique_keys)

train_keys = set(unique_keys[:int(0.7 * n)])
val_keys   = set(unique_keys[int(0.7 * n):int(0.85 * n)])
test_keys  = set(unique_keys[int(0.85 * n):])

print("Total SplitKeys:", n)
print("Train Keys:", len(train_keys))
print("Val Keys:", len(val_keys))
print("Test Keys:", len(test_keys))

# =========================================================
# STEP 4: ASSIGN SPLITS
# =========================================================

df_events["split"] = df_events["SplitKey"].apply(
    lambda x: "train" if x in train_keys
    else "val" if x in val_keys
    else "test"
)

train_df = df_events[df_events["split"] == "train"].reset_index(drop=True)
val_df   = df_events[df_events["split"] == "val"].reset_index(drop=True)
test_df  = df_events[df_events["split"] == "test"].reset_index(drop=True)

# =========================================================
# STEP 5: HARD LEAKAGE CHECK
# =========================================================

train_keys_set = set(train_df["SplitKey"])
val_keys_set   = set(val_df["SplitKey"])
test_keys_set  = set(test_df["SplitKey"])

print("\n===== LEAKAGE CHECK =====")
print("Train-Val:", len(train_keys_set & val_keys_set))
print("Train-Test:", len(train_keys_set & test_keys_set))
print("Val-Test:", len(val_keys_set & test_keys_set))

assert len(train_keys_set & val_keys_set) == 0
assert len(train_keys_set & test_keys_set) == 0
assert len(val_keys_set & test_keys_set) == 0

# =========================================================
# STEP 6: LABEL CHECK
# =========================================================

def extract_labels(df_split):
    labels = set()
    for actions in df_split["DeviceActions"]:
        for d in actions:
            labels.add(f"{d['Device']}_{d['Action']}")
    return labels

train_labels = extract_labels(train_df)
val_labels   = extract_labels(val_df)
test_labels  = extract_labels(test_df)

print("\n===== LABEL CHECK =====")
print("Val unseen labels:", len(val_labels - train_labels))
print("Test unseen labels:", len(test_labels - train_labels))

# =========================================================
# STEP 7: SAFE CONVERSION
# =========================================================

def convert(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: convert(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [convert(v) for v in obj]
    return obj

train_json = convert(train_df.to_dict("records"))
val_json   = convert(val_df.to_dict("records"))
test_json  = convert(test_df.to_dict("records"))

# =========================================================
# STEP 8: SAVE FILES
# =========================================================

with open("train.json", "w", encoding="utf-8") as f:
    json.dump(train_json, f, indent=2)

with open("val.json", "w", encoding="utf-8") as f:
    json.dump(val_json, f, indent=2)

with open("test.json", "w", encoding="utf-8") as f:
    json.dump(test_json, f, indent=2)

# =========================================================
# STEP 9: SUMMARY
# =========================================================

def summary(df):
    return {
        "events": len(df),
        "commands": df["Command"].nunique(),
        "locations": df["Location"].nunique(),
        "scenarios": df["Scenario"].nunique(),
        "intents": df["Intent_Type"].nunique(),
        "devices": df["DeviceActions"].apply(len).sum(),
        "unique_event_ids": df["Event_ID"].nunique()
    }

print("\n===== FINAL SPLIT SUMMARY =====")
print("Train:", summary(train_df))
print("Val:", summary(val_df))
print("Test:", summary(test_df))

print("\n===== DONE (FULLY LEAKAGE-FREE SPLIT) =====")

Total SplitKeys: 4920
Train Keys: 3444
Val Keys: 738
Test Keys: 738

===== LEAKAGE CHECK =====
Train-Val: 0
Train-Test: 0
Val-Test: 0

===== LABEL CHECK =====
Val unseen labels: 0
Test unseen labels: 0

===== FINAL SPLIT SUMMARY =====
Train: {'events': 140040, 'commands': 1641, 'locations': 5, 'scenarios': 23, 'intents': 2, 'devices': np.int64(521880), 'unique_event_ids': 140040}
Val: {'events': 30000, 'commands': 589, 'locations': 5, 'scenarios': 23, 'intents': 2, 'devices': np.int64(107520), 'unique_event_ids': 30000}
Test: {'events': 30280, 'commands': 604, 'locations': 5, 'scenarios': 23, 'intents': 2, 'devices': np.int64(114560), 'unique_event_ids': 30280}

===== DONE (FULLY LEAKAGE-FREE SPLIT) =====
